# 2 — Token-language toolkit and SDXL generation from token IDs

This notebook separates four operations that were previously mixed into one 1,100-line cell:

1. Search the SDXL tokenizer vocabulary.
2. Encode a prompt into token IDs.
3. Decode and preview explicit IDs.
4. Generate SDXL Turbo images from IDs.

The original notebook copied most of `StableDiffusionXLPipeline` and patched its `encode_prompt` method. That approach is fragile because it depends on internal Diffusers implementation details. The refactor instead uses the stock pipeline and supplies `prompt_embeds` and `pooled_prompt_embeds` built from the IDs.

In [ ]:
# Tokenizer-only work:
# %pip install -e "..[ui,notebooks]"

# SDXL generation as well:
# %pip install -e "..[all]"

## Tokenizer inspection (CPU-friendly)

SDXL uses two CLIP text encoders. The dictionary UI displays the **primary tokenizer**. Generation validates IDs against the shared usable range and feeds the same content IDs to both encoders, matching the experiment in the uploaded notebook.

In [ ]:
from clip_token_lab.tokens import TokenToolkit

tokens = TokenToolkit()
print("vocab size:", tokens.vocab_size)
tokens.search("astronaut", limit=10)

## Prompt → IDs

Special beginning/end tokens are omitted from the displayed content IDs. They are reinserted when building the fixed-length CLIP input sequence.

In [ ]:
prompt = "a cinematic photo of a cat astronaut"
ids = tokens.encode(prompt)
ids

## IDs → text and token preview

Tokenizer strings often contain markers for whitespace or word boundaries. The preview intentionally exposes those raw token spellings rather than pretending they are ordinary words.

In [ ]:
print(tokens.decode(ids))
tokens.entries(ids[:12])

## IDs → SDXL prompt embeddings

For each SDXL tokenizer:

1. Add BOS and EOS.
2. Pad to that tokenizer's CLIP context length.
3. Run the corresponding text encoder with hidden states enabled.
4. Concatenate the penultimate hidden states from both encoders.
5. Keep the pooled output from the second encoder.

Those tensors are passed to the unmodified Diffusers pipeline. This is much smaller and easier to update than maintaining a copied pipeline class.

In [ ]:
# Requires an NVIDIA CUDA GPU and the [all] install profile.
# from clip_token_lab.sdxl_tokens import SDXLTokenGenerator
# generator = SDXLTokenGenerator()
# images = generator.generate(ids, seed=42, steps=1, count=4)
# images[0]

## Launch the UI

Tokenizer features load first. The heavier SDXL pipeline is created only when Generate is clicked.

In [ ]:
from clip_token_lab.apps.tokens import build_demo

demo = build_demo()
demo.launch(inline=True)

## Minimal scripts

- `scripts/tokens/search_vocab.py`
- `scripts/tokens/prompt_to_ids.py`
- `scripts/tokens/ids_to_text.py`
- `scripts/tokens/ids_to_image.py`

The first three are CPU-friendly. Image generation is CUDA-only in this repository.